# Lecture 6 — Instructor Live Demo Notebook
**Runnable illustrations mapped to the updated presentation (64 slides).**

Jump to a section by slide number. Run Setup once; each demo is otherwise standalone within its group. Python snippets in the slides are sometimes abbreviated: the surrounding definitions below make them executable. All order, refund, search and payment operations are **simulated**, not live services.

**Notes:** LLM-dependent calls require a configured compatible model with tool calling and structured output. Results may vary. HITL demos use separate thread IDs to avoid stale checkpoints.

## Setup — used by all demos

In [ ]:
%pip install -q langgraph langchain-openai langchain-core python-dotenv pydantic tenacity

In [ ]:
import os
from dotenv import load_dotenv
from typing import Annotated, Literal, TypedDict
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import interrupt, Command

load_dotenv()

# Instructor-provided model settings. OPENAI_ENDPOINT is optional for a standard OpenAI endpoint.
model_name = os.getenv("OPENAI_MODEL")
model_args = {"model": model_name, "api_key": os.getenv("OPENAI_API_KEY")}
if os.getenv("OPENAI_ENDPOINT"):
    model_args["base_url"] = os.environ["OPENAI_ENDPOINT"]

assert model_args["api_key"], "Set OPENAI_API_KEY in .env (or your environment)."
llm = ChatOpenAI(**model_args)
print("Model configured:", model_name)

Model configured: model-group1


## Slides 8–11 — Simple tool calling vs ReAct
The slide examples are conceptual workflows. This live code uses mock tools so students can see *one or several tool calls* as actual LangChain messages.

In [2]:
# Safe, simulated data: no network requests
ORDERS = {
    "12345": {"status": "delayed", "expected_delivery": "2026-10-18", "amount_eur": 150},
    "56789": {"status": "shipped", "expected_delivery": "2026-10-25", "amount_eur": 45},
}

@tool
def lookup_order(order_id: str) -> str:
    """Look up a simulated customer's order by order ID."""
    return str(ORDERS.get(order_id, "Order not found"))

@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

tools_e1 = [lookup_order, add_numbers]
agent_model_e1 = llm.bind_tools(tools_e1)
print([t.name for t in tools_e1])

['lookup_order', 'add_numbers']


In [3]:
# Single tool call, with an explicit tool-bound model
single_model = llm.bind_tools([lookup_order])
msg = single_model.invoke([HumanMessage(content="Look up order 12345 with the tool.")])
print("Tool calls:", msg.tool_calls)
if msg.tool_calls:
    print("Tool result:", lookup_order.invoke(msg.tool_calls[0]["args"]))


Tool calls: [{'name': 'lookup_order', 'args': {'order_id': '12345'}, 'id': 'call_ykmGGLnBQJSigVKZKU52GxrE', 'type': 'tool_call'}]
Tool result: {'status': 'delayed', 'expected_delivery': '2026-10-18', 'amount_eur': 150}


## Slides 19–23 and 28 — LangGraph state, nodes, edges, conditional edges
Shows `MessagesState` (built using `add_messages`), node partial updates and routing. The `should_continue` function is a **router**, not a graph node.

In [4]:
def demo_agent(state: MessagesState):
    answer = demo_agent_model.invoke([
        SystemMessage(content="You are a concise customer-support assistant. Use tools when needed."),
        *state["messages"],
    ])
    return {"messages": [answer]}

def should_continue_demo(state: MessagesState) -> Literal["tools", "end"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

In [5]:
demo_agent_model = llm.bind_tools([lookup_order, add_numbers])
# agent function above resolves demo_agent_model when invoked.
g = StateGraph(MessagesState)
g.add_node("agent", demo_agent)
g.add_node("tools", ToolNode([lookup_order, add_numbers]))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_continue_demo, {"tools": "tools", "end": END})
g.add_edge("tools", "agent")
agent_app_demo = g.compile()
print("Graph built")
result = agent_app_demo.invoke({"messages": [HumanMessage(content="Look up order 12345.")]})
for msg in result["messages"]:
    print(type(msg).__name__, str(msg.content)[:140], getattr(msg,"tool_calls",[]))


Graph built
HumanMessage Look up order 12345. []
AIMessage  [{'name': 'lookup_order', 'args': {'order_id': '12345'}, 'id': 'call_sGROFNozDSj9R9FinXma8CJ0', 'type': 'tool_call'}]
ToolMessage {'status': 'delayed', 'expected_delivery': '2026-10-18', 'amount_eur': 150} []
AIMessage Order 12345 is delayed. Expected delivery: 2026-10-18. Amount: €150. []


## Slide 24 — Error handling and exponential backoff
Retries only simulated transient exceptions, and does not retry other failures.

In [6]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
class TransientFailure(Exception): pass
tries = {"n": 0}
@retry(stop=stop_after_attempt(3),wait=wait_exponential(multiplier=.1,min=.1,max=.5),retry=retry_if_exception_type(TransientFailure),reraise=True)
def flaky_api():
    tries["n"] += 1
    print("Attempt", tries["n"])
    if tries["n"] < 3: raise TransientFailure("temporary")
    return "OK"
print(flaky_api())


Attempt 1
Attempt 2
Attempt 3
OK


## Slides 25–27 — Human-in-the-loop with interrupt and resume
Run the pause cell before the resume cell. This checks **both** human decisions without invoking an actual payment API.

In [8]:
class RefundState(TypedDict):
    order_id: str
    amount_eur: float
    approved: bool | None
    outcome: str

def review_e1b(state: RefundState):
    approved = interrupt({
        "question": "Approve this simulated refund?",
        "order_id": state["order_id"],
        "amount_eur": state["amount_eur"],
    })
    return {"approved": bool(approved)}

def route_approval_e1b(state: RefundState) -> Literal["approve", "reject"]:
    return "approve" if state["approved"] is True else "reject"

def approve_e1b(state: RefundState):
    return {"outcome": f"SIMULATED refund approved for order {state['order_id']}"}

def reject_e1b(state: RefundState):
    return {"outcome": "No refund executed"}

builder_e1b = StateGraph(RefundState)
builder_e1b.add_node("review", review_e1b)
builder_e1b.add_node("approve", approve_e1b)
builder_e1b.add_node("reject", reject_e1b)
builder_e1b.add_edge(START, "review")
builder_e1b.add_conditional_edges("review", route_approval_e1b, {"approve": "approve", "reject": "reject"})
builder_e1b.add_edge("approve", END)
builder_e1b.add_edge("reject", END)
app_e1b = builder_e1b.compile(checkpointer=InMemorySaver())

In [9]:
for key, approve in [("instructor-refund-approve",True),("instructor-refund-reject",False)]:
    cfg = {"configurable":{"thread_id":key}}
    paused = app_e1b.invoke({"order_id":"12345","amount_eur":150.0,"approved":None,"outcome":""},cfg)
    print("Human prompt:", paused["__interrupt__"])
    resumed = app_e1b.invoke(Command(resume=approve),cfg)
    print("Outcome:",resumed["outcome"])


Human prompt: [Interrupt(value={'question': 'Approve this simulated refund?', 'order_id': '12345', 'amount_eur': 150.0}, id='6ac4d276993390164dcf8c8423cd827a', response_schema=None)]
Outcome: SIMULATED refund approved for order 12345
Human prompt: [Interrupt(value={'question': 'Approve this simulated refund?', 'order_id': '12345', 'amount_eur': 150.0}, id='378338d9e01875a9607e4145a1de8df7', response_schema=None)]
Outcome: No refund executed


## Slides 32–34 — Supervisor and structured routing
The supervisor produces a constrained `Literal` rather than free-form agent names. The researcher uses a simulated static fact; no web search occurs.

In [10]:
class TeamState(TypedDict):
    task: str
    route: str
    output: str

class RouteDecision(BaseModel):
    next_agent: Literal["researcher", "writer"]

supervisor_model_e2 = llm.with_structured_output(RouteDecision)

def supervisor_e2(state: TeamState):
    decision = supervisor_model_e2.invoke([
        ("system", "Route information-gathering requests to researcher and drafting requests to writer. Output only the structured choice."),
        ("human", state["task"]),
    ])
    return {"route": decision.next_agent}

def researcher_e2(state: TeamState):
    # No live search: findings are a simulated reference.
    return {"output": "SIMULATED RESEARCH: LangGraph uses nodes, edges and state to coordinate execution."}

def writer_e2(state: TeamState):
    answer = llm.invoke([
        ("system", "Write a short, accessible answer. Don't claim to have searched the web."),
        ("human", state["task"]),
    ])
    return {"output": str(answer.content)}

def route_team_e2(state: TeamState) -> Literal["researcher", "writer"]:
    return state["route"]

In [11]:
builder_e2 = StateGraph(TeamState)
builder_e2.add_node("supervisor", supervisor_e2)
builder_e2.add_node("researcher", researcher_e2)
builder_e2.add_node("writer", writer_e2)
builder_e2.add_edge(START, "supervisor")
builder_e2.add_conditional_edges("supervisor", route_team_e2, {
    "researcher": "researcher", "writer": "writer"
})
builder_e2.add_edge("researcher", END)
builder_e2.add_edge("writer", END)
app_e2 = builder_e2.compile()

for task in ["Gather facts about LangGraph state", "Draft a friendly introduction to Python dictionaries"]:
    result = app_e2.invoke({"task": task, "route": "", "output": ""})
    print("Route:", result["route"])
    print("Output:", result["output"])


Route: researcher
Output: SIMULATED RESEARCH: LangGraph uses nodes, edges and state to coordinate execution.
Route: writer
Output: Sure! Here’s a friendly introduction:

---

**Python dictionaries** are a way to store information in **key-value pairs**.  
Think of them like a real dictionary or a phonebook: you look up a **key** and get its **value**.

For example:

```python
person = {
    "name": "Ava",
    "age": 28,
    "city": "Paris"
}
```

In this dictionary, `"name"` is a key, and `"Ava"` is its value.

You can use dictionaries when you want to:
- organize related data
- quickly find a value by its key
- store things like user profiles, settings, or product details

A few handy things to know:
- Dictionaries are written with **curly braces** `{ }`
- Each key and value are separated by a **colon** `:`
- Pairs are separated by **commas** `,`

You can access a value like this:

```python
print(person["name"])  # Ava
```

Dictionaries are flexible, useful, and one of Python’s most 

## Slides 35–40 — Sequential pipeline and shared data
Contrast a fixed Writer → Reviewer sequence with supervisor routing.

In [12]:
class ReviewState(TypedDict):
    topic: str
    draft: str
    review: str

def writer_demo(state: ReviewState):
    return {"draft": f"An introductory explanation of {state['topic']}."}

def reviewer_demo(state: ReviewState):
    return {"review": "Check the example and add a clear practical use case."}

pipeline_demo = StateGraph(ReviewState)
pipeline_demo.add_node("writer", writer_demo)
pipeline_demo.add_node("reviewer", reviewer_demo)
pipeline_demo.add_edge(START,"writer")
pipeline_demo.add_edge("writer","reviewer")
pipeline_demo.add_edge("reviewer",END)
pipeline_app_demo = pipeline_demo.compile()
print(pipeline_app_demo.invoke({"topic":"Python dictionaries","draft":"","review":""}))


{'topic': 'Python dictionaries', 'draft': 'An introductory explanation of Python dictionaries.', 'review': 'Check the example and add a clear practical use case.'}


## Slides 45–48 — Conversation memory concepts: history, sliding window and summary
The list operations demonstrate conceptual context management, not legacy LangChain memory APIs.

In [13]:
history = ["I'm a beginner", "I learn Python", "I prefer examples", "Explain loops", "Show a for loop"]
print("Full:", history)
print("Sliding window (last 2):", history[-2:])
summary = "Student is a Python beginner who prefers concrete examples."
print("Summary plus recent:", [summary] + history[-2:])


Full: ["I'm a beginner", 'I learn Python', 'I prefer examples', 'Explain loops', 'Show a for loop']
Sliding window (last 2): ['Explain loops', 'Show a for loop']
Summary plus recent: ['Student is a Python beginner who prefers concrete examples.', 'Explain loops', 'Show a for loop']


## Slides 49–51 — Checkpointing and isolated threads

In [14]:
def learning_agent_e3(state: MessagesState):
    response = llm.invoke([
        SystemMessage(content="You are a supportive Python learning assistant. Answer concisely and use the supplied conversation context."),
        *state["messages"],
    ])
    return {"messages": [response]}

builder_e3 = StateGraph(MessagesState)
builder_e3.add_node("learning_agent", learning_agent_e3)
builder_e3.add_edge(START, "learning_agent")
app_e3 = builder_e3.compile(checkpointer=InMemorySaver())

thread_a = {"configurable": {"thread_id": "learning-chat-A"}}
thread_b = {"configurable": {"thread_id": "learning-chat-B"}}

app_e3.invoke({"messages": [HumanMessage(content="My name is Alice. I'm learning Python loops.")]}, thread_a)
response_a = app_e3.invoke({"messages": [HumanMessage(content="What topic am I learning?")]}, thread_a)
print("Same thread:", response_a["messages"][-1].content)
response_b = app_e3.invoke({"messages": [HumanMessage(content="What topic am I learning?")]}, thread_b)
print("Different thread:", response_b["messages"][-1].content)

Same thread: You’re learning **Python loops**.
Different thread: You’re learning **Python**.


## Slides 52, 54–55 — Long-term structured memory and cross-thread retrieval

In [15]:
store_e3 = InMemoryStore()
user_id_e3 = "demo-student-alice"
namespace_e3 = ("student_preferences", user_id_e3)

# Save a structured preference — no embedding model is required.
store_e3.put(namespace_e3, "preferred_language", {"language": "Python"})

stored_e3 = store_e3.get(namespace_e3, "preferred_language")
print("Retrieved long-term preference:", stored_e3.value)

Retrieved long-term preference: {'language': 'Python'}


In [16]:
def personalized_e3(state: MessagesState):
    # A real application would obtain user_id from authenticated context.
    memory = store_e3.get(namespace_e3, "preferred_language")
    preference = memory.value["language"] if memory else "unknown"
    reply = llm.invoke([
        SystemMessage(content=f"You are a programming tutor. Student's saved preferred language: {preference}."),
        *state["messages"],
    ])
    return {"messages": [reply]}

builder_e3_long = StateGraph(MessagesState)
builder_e3_long.add_node("personalized", personalized_e3)
builder_e3_long.add_edge(START, "personalized")
app_e3_long = builder_e3_long.compile(checkpointer=InMemorySaver())

new_thread_e3 = {"configurable": {"thread_id": "learning-chat-C"}}
result = app_e3_long.invoke({"messages": [HumanMessage(content="Which programming language do I prefer?")]}, new_thread_e3)
print(result["messages"][-1].content)

You prefer **Python**.


## Slide 53 — Vector memory and similarity search (optional)
Requires an embedding provider and `langchain-chroma`. The slide explains a *concept*; the example below uses local ephemeral Chroma and should not be confused with exact structured lookup. Skip if class credentials/packages are unavailable.

In [ ]:
# OPTIONAL — uncomment to demonstrate embeddings-backed similarity search.
# %pip install -q langchain-chroma
# from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
# vs = Chroma(collection_name="lecture6_demo", embedding_function=OpenAIEmbeddings())
# vs.add_texts(["Student struggled with Python loops", "Student enjoys Italian food"], metadatas=[{"user_id":"demo"},{"user_id":"demo"}])
# for doc in vs.similarity_search("What programming topic was difficult?", k=1, filter={"user_id":"demo"}):
#     print(doc.page_content)


## Slides 56–58 — Safety and architecture recap
A checkpointer stores *per-thread graph state*. A store keeps *cross-thread application information*. Neither automatically authorizes one user to access another user’s data.

**Discussion prompts:** Which tool calls need approval? Why is an LLM-only reviewer insufficient for validation? When is a deterministic function better than another agent?